# Exact match or string matching measure (using fuzzy logic)

In [2]:
from google.colab import drive;
drive.mount('/content/drive');
!pip install pandas


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!pip install -U scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 46.0 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.3.2
    Uninstalling scikit-learn-1.3.2:
      Successfully uninstalled scikit-learn-1.3.2


In [5]:
!pip install fuzzywuzzy pandas openpyxl

In [6]:
import pandas as pd

file_path = '/content/drive/MyDrive/Colab Notebooks/IST/GPT_4o/Result/R2/R2.xlsx'
df = pd.read_excel(file_path)
print(df.head())

     Run                                            ACAS XU  \
0  Run 1  G0: ACAS Xu satisfies security requirements\nG...   
1  Run 2  G0: ACAS Xu satisfies security requirements\nG...   
2  Run 3  G0: ACAS Xu satisfies security requirements\nG...   
3  Run 4  G0: ACAS Xu satisfies security requirements\nG...   
4  Run 5  G0: ACAS Xu satisfies security requirements.\n...   

                                            BlueROV2  \
0  G1: Hazards for BlueROV2 system addressed in a...   
1  G1: Hazards for the BlueROV2 system addressed ...   
2  G1: Hazards for BlueROV2 system addressed in a...   
3  G1: Hazards for BlueROV2 system addressed in a...   
4  G1: Hazards for BlueROV2 system addressed in a...   

                                                GPCA  \
0  G1: Operational safety is verified in GPCA sys...   
1  G1: Operational safety is verified in GPCA sys...   
2  G1: Operational safety is verified in Generic ...   
3  G1: Operational safety is verified in the GPCA...   
4  G

In [7]:
file_path2 = '/content/drive/MyDrive/Colab Notebooks/IST/GPT_4o/Result/Ground_Truth_AC.xlsx'
ground_truth_df = pd.read_excel(file_path2)
print(ground_truth_df.head())

          Case                                                 AC
0      ACAS XU  G0: ACAS Xu satisfies security requirements\nG...
1     BlueROV2  G1: Hazards for BlueROV2 addressed in accordan...
2         GPCA  G1: Operational safety is verified in "GPCA sy...
3  IM_Software  G1: IM server software is acceptable secure in...


In [8]:
from fuzzywuzzy import fuzz


for index, experiment_row in df.iterrows():
    for case in ['ACAS XU', 'BlueROV2', 'GPCA', 'IM_Software']:
        experiment_text = experiment_row[case]
        ground_truth_text = ground_truth_df.loc[ground_truth_df['Case'] == case, 'AC'].iloc[0]
        similarity_score = fuzz.ratio(experiment_text, ground_truth_text)/ 100.0

        print(f"Similarity score for {case} in {experiment_row['Run']}: {similarity_score}")
    print("-------")

Similarity score for ACAS XU in Run 1: 0.73
Similarity score for BlueROV2 in Run 1: 0.64
Similarity score for GPCA in Run 1: 0.22
Similarity score for IM_Software in Run 1: 0.1
-------
Similarity score for ACAS XU in Run 2: 0.89
Similarity score for BlueROV2 in Run 2: 0.65
Similarity score for GPCA in Run 2: 0.26
Similarity score for IM_Software in Run 2: 0.34
-------
Similarity score for ACAS XU in Run 3: 0.85
Similarity score for BlueROV2 in Run 3: 0.19
Similarity score for GPCA in Run 3: 0.29
Similarity score for IM_Software in Run 3: 0.1
-------
Similarity score for ACAS XU in Run 4: 0.79
Similarity score for BlueROV2 in Run 4: 0.77
Similarity score for GPCA in Run 4: 0.23
Similarity score for IM_Software in Run 4: 0.1
-------
Similarity score for ACAS XU in Run 5: 0.61


/usr/local/lib/python3.10/dist-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


Similarity score for BlueROV2 in Run 5: 0.77
Similarity score for GPCA in Run 5: 0.31
Similarity score for IM_Software in Run 5: 0.32
-------


# Bleu score

In [9]:
!pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 4.7 MB/s eta 0:00:00


In [10]:
import sacrebleu

def calculate_bleu_score(experiment_text, ground_truth_text):
    references = [[ground_truth_text]]
    system_output = [experiment_text]
    bleu = sacrebleu.corpus_bleu(system_output, references)
    # Scale the score to a range between 0 and 1 and round to two decimal points
    return round(bleu.score / 100, 2)

for case in ['ACAS XU', 'BlueROV2', 'GPCA', 'IM_Software']:
    print(f"Calculating BLEU scores for {case}:")
    for index, row in df.iterrows():
        experiment_text = row[case]
        ground_truth_text = ground_truth_df.loc[ground_truth_df['Case'] == case, 'AC'].iloc[0]

        if pd.notnull(experiment_text) and pd.notnull(ground_truth_text):
            bleu_score = calculate_bleu_score(experiment_text, ground_truth_text)
            print(f"Run {index + 1} BLEU score: {bleu_score}")
        else:
            print(f"Run {index + 1} BLEU score: Data Missing")
    print("-------")


Calculating BLEU scores for ACAS XU:
Run 1 BLEU score: 0.73
Run 2 BLEU score: 0.83
Run 3 BLEU score: 0.7
Run 4 BLEU score: 0.79
Run 5 BLEU score: 0.48
-------
Calculating BLEU scores for BlueROV2:
Run 1 BLEU score: 0.54
Run 2 BLEU score: 0.62
Run 3 BLEU score: 0.4
Run 4 BLEU score: 0.49
Run 5 BLEU score: 0.61
-------
Calculating BLEU scores for GPCA:
Run 1 BLEU score: 0.27
Run 2 BLEU score: 0.33
Run 3 BLEU score: 0.35
Run 4 BLEU score: 0.32
Run 5 BLEU score: 0.26
-------
Calculating BLEU scores for IM_Software:
Run 1 BLEU score: 0.22
Run 2 BLEU score: 0.27
Run 3 BLEU score: 0.29
Run 4 BLEU score: 0.28
Run 5 BLEU score: 0.27
-------


# Semantic Similarity

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()

for case in ['ACAS XU', 'BlueROV2', 'GPCA', 'IM_Software']:
    experiment_texts = df[case].tolist()
    ground_truth_text = ground_truth_df.loc[ground_truth_df['Case'] == case, 'AC'].iloc[0]

    combined_texts = experiment_texts + [ground_truth_text]

    tfidf_matrix = vectorizer.fit_transform(combined_texts)

    experiment_vectors = tfidf_matrix[:-1]
    ground_truth_vector = tfidf_matrix[-1:]

    cosine_similarities = cosine_similarity(experiment_vectors, ground_truth_vector)

    for i, similarity_score in enumerate(cosine_similarities.flatten(), 1):
        # Round the score to two decimal places without multiplying by 100
        rounded_score = round(similarity_score, 2)
        print(f"Cosine similarity for {case}, Run {i}: {rounded_score}")
    print("-------")


Cosine similarity for ACAS XU, Run 1: 0.86
Cosine similarity for ACAS XU, Run 2: 0.94
Cosine similarity for ACAS XU, Run 3: 0.92
Cosine similarity for ACAS XU, Run 4: 0.93
Cosine similarity for ACAS XU, Run 5: 0.89
-------
Cosine similarity for BlueROV2, Run 1: 0.91
Cosine similarity for BlueROV2, Run 2: 0.88
Cosine similarity for BlueROV2, Run 3: 0.88
Cosine similarity for BlueROV2, Run 4: 0.89
Cosine similarity for BlueROV2, Run 5: 0.83
-------
Cosine similarity for GPCA, Run 1: 0.67
Cosine similarity for GPCA, Run 2: 0.81
Cosine similarity for GPCA, Run 3: 0.82
Cosine similarity for GPCA, Run 4: 0.84
Cosine similarity for GPCA, Run 5: 0.68
-------
Cosine similarity for IM_Software, Run 1: 0.7
Cosine similarity for IM_Software, Run 2: 0.68
Cosine similarity for IM_Software, Run 3: 0.7
Cosine similarity for IM_Software, Run 4: 0.71
Cosine similarity for IM_Software, Run 5: 0.71
-------
